# 🌐 FOMV: Explorador Interactivo del Cubo Multivariable

**Autor:** Osvaldo Morales  
**Licencia:** AGPL-3.0 (código), CC BY-NC-ND 4.0 (documentación)

---

### 🧠 ¿Qué hace este notebook?
1. **Simula** el modelo HARD‑nonlinear en un grid de (Backlog B, Memory M).
2. **Calcula** el tiempo medio al colapso (MFPT) y promedios de variables rápidas (E, G, T, C).
3. **Visualiza** los resultados en:
   - Un gráfico de contorno 2D (B vs M).
   - Un **cubo 3D interactivo** donde puedes elegir qué variables representar en los ejes X, Y, Z y el color.
   - **Cortes 2D dinámicos** con sliders para fijar una variable.

---

### ⚙️ Parámetros de simulación (puedes modificarlos)
Los valores actuales están ajustados para una ejecución rápida (~1 minuto). Para más resolución, aumenta `Bgrid` y `Mgrid`.

In [ ]:
# Instalación rápida de dependencias (solo necesario en Colab)
!pip install -q pandas plotly ipywidgets

In [ ]:
"""
FOMV: Field Operator for Measured Viability - Versión Didáctica
Autor: Osvaldo Morales
License: AGPL-3.0
"""

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import multiprocessing as mp
from functools import partial
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# PARÁMETROS DEL MODELO (no modificar a menos que conozcas la dinámica)
# -----------------------------------------------------------------------------
params = {
    'alpha1': 0.1, 'alpha2': 0.2, 'delta': 0.05, 'beta1': 0.3,
    'gamma1': 0.2, 'gamma2': 0.1, 'gamma3': 0.1, 'phi1': 0.3,
    'phi2': 0.2, 'psi1': 0.2, 'psi2': 0.2, 'kappa1': 0.2, 'kappa2': 0.1,
    'Ec': 0.1, 'Er': 0.5, 'Lc': 1.5, 'Lr': 0.8,
}

# -----------------------------------------------------------------------------
# PARÁMETROS DE SIMULACIÓN (ajusta según necesidad)
# -----------------------------------------------------------------------------
sim_params = {
    'sigma': 0.05,
    'Tmax': 50,
    'R': 500,
    'Bgrid': 20,
    'Mgrid': 20,
    'B_range': [0, 1.2],
    'M_range': [0, 0.8],
    'bootstrap_reps': 100,
    'alpha': 0.05,
    'fast_samples': 20,
    'n_cores': mp.cpu_count(),
    'seed': 42
}

# -----------------------------------------------------------------------------
# FUNCIONES DE DINÁMICA (optimizadas)
# -----------------------------------------------------------------------------
def sigmoid(x): return 1/(1+np.exp(-x))

def hard_nonlinear_dynamics_vectorized(x, theta, eta):
    B, M, E, G, T, C = x[:,0], x[:,1], x[:,2], x[:,3], x[:,4], x[:,5]
    B_star = B + theta['alpha1']*(1 - E) - theta['alpha2']*G
    M_star = (1 - theta['delta'])*M + theta['beta1'] * sigmoid(B - T)
    E_star = E + theta['gamma1']*G - theta['gamma2']*B - theta['gamma3']*M
    G_star = G + theta['phi1']*E - theta['phi2']*(B + M)*(1 - T)
    T_star = T - theta['psi1']*M*(1 - G) + theta['psi2']*G
    C_star = C + theta['kappa1']*T - theta['kappa2']*B
    x_star = np.column_stack([B_star, M_star, E_star, G_star, T_star, C_star]) + eta
    return np.clip(x_star, 0, 1)

def generate_noise_vectorized(sigma, n):
    beta = np.random.beta(2, 2, size=(n, 6))
    u = 2 * beta - 1
    return sigma * u

def is_collapsed_vectorized(x, theta):
    B, M, E = x[:,0], x[:,1], x[:,2]
    return (E <= theta['Ec']) | (B + M >= theta['Lc'])

def is_recovered_vectorized(x, theta):
    B, M, E = x[:,0], x[:,1], x[:,2]
    return (E >= theta['Er']) & (B + M <= theta['Lr'])

def simulate_trajectories_vectorized(x0, theta, sigma, Tmax):
    n = x0.shape[0]
    x = x0.copy()
    absorptions = np.full(n, None, dtype=object)
    times = np.full(n, Tmax, dtype=int)
    active = np.ones(n, dtype=bool)
    for t in range(Tmax):
        if not np.any(active): break
        collapsed = is_collapsed_vectorized(x[active], theta)
        recovered = is_recovered_vectorized(x[active], theta)
        idx_active = np.where(active)[0]
        absorptions[idx_active[collapsed]] = 'C'
        times[idx_active[collapsed]] = t
        absorptions[idx_active[recovered]] = 'R'
        times[idx_active[recovered]] = t
        active[idx_active[collapsed]] = False
        active[idx_active[recovered]] = False
        if not np.any(active): break
        eta = generate_noise_vectorized(sigma, np.sum(active))
        x[active] = hard_nonlinear_dynamics_vectorized(x[active], theta, eta)
    return absorptions, times

def generate_fast_samples(B, M, theta, sigma, n_samples, burnin=500, seed=None):
    if seed is not None:
        np.random.seed(seed)
    samples = []
    fast = np.random.uniform(0, 1, size=4)
    total_steps = n_samples + burnin
    for i in range(total_steps):
        x = np.array([B, M, fast[0], fast[1], fast[2], fast[3]])
        eta = generate_noise_vectorized(sigma, 1)[0]
        x_new = hard_nonlinear_dynamics_vectorized(x.reshape(1,-1), theta, eta.reshape(1,-1))
        fast = x_new[0, 2:]
        if i >= burnin:
            samples.append(fast.copy())
    samples = np.array(samples)
    return samples, (np.mean(samples[:,0]), np.mean(samples[:,1]),
                     np.mean(samples[:,2]), np.mean(samples[:,3]))

def compute_point(BM, theta, sigma, Tmax, R, fast_samples, base_seed):
    B, M = BM
    seed = base_seed + int(B * 10000 + M * 1000)  # determinista
    np.random.seed(seed)
    try:
        fast_arr, (Ea, Ga, Ta, Ca) = generate_fast_samples(B, M, theta, sigma, fast_samples)
        all_times_C = []
        q_sum = 0.0
        total_traj = 0
        for fast in fast_arr:
            x0 = np.tile(np.array([B, M, fast[0], fast[1], fast[2], fast[3]]), (R, 1))
            absorptions, times = simulate_trajectories_vectorized(x0, theta, sigma, Tmax)
            q_sum += np.sum(absorptions == 'R')
            total_traj += R
            all_times_C.extend(times[absorptions == 'C'])
        q_hat = q_sum / total_traj if total_traj>0 else np.nan
        mfpt_hat = np.mean(all_times_C) if all_times_C else np.nan
        return (B, M, q_hat, mfpt_hat, all_times_C, Ea, Ga, Ta, Ca)
    except Exception as e:
        print(f"Error en ({B:.2f},{M:.2f}): {e}")
        return (B, M, np.nan, np.nan, [], np.nan, np.nan, np.nan, np.nan)

def estimate_on_grid_parallel(B_grid, M_grid, theta, sigma, Tmax, R,
                              fast_samples, n_cores, base_seed):
    points = [(B, M) for B in B_grid for M in M_grid]
    func = partial(compute_point, theta=theta, sigma=sigma,
                   Tmax=Tmax, R=R, fast_samples=fast_samples,
                   base_seed=base_seed)
    with mp.Pool(processes=n_cores) as pool:
        results = list(tqdm(pool.imap(func, points), total=len(points), desc="Puntos de grid"))
    nB, nM = len(B_grid), len(M_grid)
    Q = np.full((nB, nM), np.nan)
    MFPT = np.full((nB, nM), np.nan)
    E = np.full((nB, nM), np.nan); G = np.full((nB, nM), np.nan)
    T = np.full((nB, nM), np.nan); C = np.full((nB, nM), np.nan)
    times_data = {}
    idx = 0
    for i, B in enumerate(B_grid):
        for j, M in enumerate(M_grid):
            (_, _, q, mfpt, times, Ea, Ga, Ta, Ca) = results[idx]
            Q[i,j] = q
            MFPT[i,j] = mfpt
            E[i,j] = Ea; G[i,j] = Ga; T[i,j] = Ta; C[i,j] = Ca
            times_data[(i,j)] = times
            idx += 1
    return Q, MFPT, times_data, E, G, T, C

# -----------------------------------------------------------------------------
# EJECUCIÓN DE LA SIMULACIÓN
# -----------------------------------------------------------------------------
print("="*60)
print("🔬 FOMV: Simulación en curso...")
print("="*60)

B_grid = np.linspace(sim_params['B_range'][0], sim_params['B_range'][1], sim_params['Bgrid'])
M_grid = np.linspace(sim_params['M_range'][0], sim_params['M_range'][1], sim_params['Mgrid'])

print(f"\n🧮 Grid: {sim_params['Bgrid']}×{sim_params['Mgrid']} puntos")
print(f"⚙️  Parámetros: R={sim_params['R']}, fast_samples={sim_params['fast_samples']}, sigma={sim_params['sigma']}")
print(f"🖥️  Usando {sim_params['n_cores']} núcleos\n")

Q, MFPT, times_data, E, G, T, C = estimate_on_grid_parallel(
    B_grid, M_grid, params, sim_params['sigma'],
    sim_params['Tmax'], sim_params['R'],
    sim_params['fast_samples'], sim_params['n_cores'],
    sim_params['seed']
)

print("\n✅ Simulación completada.")

## 📊 Resultados básicos

A continuación, el gráfico de contorno muestra el tiempo medio al colapso (MFPT) en función de **Backlog B** y **Memory M**. Zonas más claras indican mayor estabilidad (mayor MFPT).

In [ ]:
plt.figure(figsize=(8,6))
B_mesh, M_mesh = np.meshgrid(B_grid, M_grid, indexing='ij')
contour = plt.contourf(B_mesh, M_mesh, MFPT, levels=20, cmap='viridis')
plt.colorbar(contour, label='MFPT (tiempo de colapso)')
plt.xlabel('Backlog B'); plt.ylabel('Memory M')
plt.title('MFPT en el plano (B, M)')
plt.show()

## 🧩 Construcción del DataFrame

Combinamos todas las variables en una tabla para facilitar la exploración.

In [ ]:
B_vals = np.repeat(B_grid, len(M_grid))
M_vals = np.tile(M_grid, len(B_grid))
df = pd.DataFrame({
    'B': B_vals,
    'M': M_vals,
    'E': E.flatten(),
    'G': G.flatten(),
    'T': T.flatten(),
    'C': C.flatten(),
    'MFPT': MFPT.flatten(),
    'Q': Q.flatten()
}).dropna()

# Añadir log(MFPT) para mejor visualización
df['logMFPT'] = np.log(df['MFPT'].clip(lower=1e-10))

print(f"📋 DataFrame con {len(df)} puntos válidos")
print(f"Columnas disponibles: {list(df.columns)}")
df.head()

## 🎮 Explorador interactivo del cubo

Selecciona las variables para los ejes X, Y, Z y el color. El gráfico 3D se actualizará al hacer clic en el botón.

In [ ]:
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

var_options = ['B', 'M', 'E', 'G', 'T', 'C', 'MFPT', 'Q', 'logMFPT']

# Widgets
x_w = widgets.Dropdown(options=var_options, value='B', description='X:')
y_w = widgets.Dropdown(options=var_options, value='M', description='Y:')
z_w = widgets.Dropdown(options=var_options, value='T', description='Z:')
color_w = widgets.Dropdown(options=var_options, value='logMFPT', description='Color:')
size_w = widgets.FloatSlider(value=3, min=1, max=10, step=0.5, description='Tamaño:')
button = widgets.Button(description='Actualizar gráfico')
out = widgets.Output()

def update_3d(b):
    with out:
        out.clear_output()
        fig = px.scatter_3d(df, x=x_w.value, y=y_w.value, z=z_w.value,
                            color=color_w.value, color_continuous_scale='viridis',
                            title=f'{x_w.value} vs {y_w.value} vs {z_w.value}',
                            opacity=0.8)
        fig.update_traces(marker=dict(size=size_w.value))
        fig.show()

button.on_click(update_3d)
display(widgets.VBox([widgets.HBox([x_w, y_w, z_w]),
                      widgets.HBox([color_w, size_w]),
                      button, out]))

## 🔪 Cortes 2D interactivos

Fija una variable (por ejemplo, M) y mueve el deslizador para ver cómo varían las otras dos en ese corte.

In [ ]:
from ipywidgets import interact

var_2d = ['B', 'M', 'E', 'G', 'T', 'C']

@interact(
    variable_fija=widgets.Dropdown(options=var_2d, value='M', description='Fijar:'),
    valor=widgets.FloatSlider(min=df['M'].min(), max=df['M'].max(), step=0.05, description='Valor:'),
    x=widgets.Dropdown(options=var_2d, value='B', description='Eje X:'),
    y=widgets.Dropdown(options=var_2d, value='T', description='Eje Y:'),
    color=widgets.Dropdown(options=['MFPT', 'logMFPT', 'Q'], value='logMFPT', description='Color:')
)
def plot_slice(variable_fija, valor, x, y, color):
    tol = 0.05 * (df[variable_fija].max() - df[variable_fija].min())
    subset = df[np.abs(df[variable_fija] - valor) < tol]
    if len(subset) == 0:
        print("No hay datos en ese corte")
        return
    plt.figure(figsize=(8,6))
    sc = plt.scatter(subset[x], subset[y], c=subset[color],
                     cmap='viridis', s=50, edgecolor='k')
    plt.colorbar(sc, label=color)
    plt.xlabel(x); plt.ylabel(y)
    plt.title(f'{variable_fija} ≈ {valor:.2f}')
    plt.grid(True)
    plt.show()

## 📁 Exportar datos

Puedes guardar el DataFrame como CSV para análisis posteriores.

In [ ]:
# Descomenta la siguiente línea para guardar
# df.to_csv('resultados_fomv.csv', index=False)
print("Archivo guardado (si descomentaste la línea).")

## ✅ Proceso finalizado

¡Ya tienes todo listo para explorar tu modelo de forma interactiva! Comparte este notebook con quien quieras.